# Fuga de datos

Una **fuga** (*leakage*) es información que llega a un modelo durante el entrenamiento, o durante su evaluación, y que no tendrá cuando se use. El resultado es siempre el mismo: una puntuación mejor de la que merece el modelo, y un modelo que decepciona en producción. Una fuga rara vez rompe algo de forma visible; el código se ejecuta, los números parecen buenos, y eso es lo que la hace peligrosa.

A continuación, cuatro demostraciones breves, cada una con la puntuación con fuga junto a la honesta. Un quinto tipo, elegir un modelo por su puntuación de prueba, se midió en [Validación y selección de modelos](validation_and_model_selection.ipynb#elegir-con-el-conjunto-de-prueba).

## Aprender de las filas de validación

Cualquier paso que aprende de los datos, un escalador, un imputador, una selección de variables, debe aprender solo de las filas de entrenamiento ([el conjunto de prueba representa la producción](overfitting_and_the_split.ipynb#el-conjunto-de-prueba-representa-la-producción)). En un escalador, aprender de todas las filas cambia muy poco el resultado, por eso el error pasa desapercibido. En la selección de variables puede cambiarlo todo.

Un caso extremo: 100 filas con 10 000 variables de puro ruido aleatorio, y un objetivo que también es ruido aleatorio. Ningún modelo puede predecirlo; el R² honesto es de alrededor de 0. Seleccionamos las 20 variables más correlacionadas con el objetivo *antes* de la validación cruzada, usando todas las filas:

In [ ]:
import numpy as np
from sklearn.feature_selection import SelectKBest, f_regression
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import KFold, cross_val_score
from sklearn.pipeline import make_pipeline

rng = np.random.default_rng(0)
X_noise = rng.normal(size=(100, 10_000))
y_noise = rng.normal(size=100)
folds = KFold(n_splits=5, shuffle=True, random_state=0)

selected_with_every_row = SelectKBest(f_regression, k=20).fit_transform(X_noise, y_noise)
cross_val_score(LinearRegression(), selected_with_every_row, y_noise, cv=folds, scoring="r2").mean()

np.float64(0.6082550151604569)

Y con la selección dentro del pipeline, para que cada ronda seleccione sus variables a partir de sus propios pliegues de entrenamiento:

In [ ]:
selection_inside = make_pipeline(SelectKBest(f_regression, k=20), LinearRegression())
cross_val_score(selection_inside, X_noise, y_noise, cv=folds, scoring="r2").mean()

np.float64(-0.0306497038602491)

Un R² de 0,61 con datos que no contienen ninguna información, frente a alrededor de 0 cuando cada ronda selecciona sus propias variables. Entre 10 000 variables aleatorias, algunas resultan estar correlacionadas con este objetivo aleatorio concreto. Elegirlas con todas las filas, las de validación incluidas, escoge justo las que resultan ajustarse también a las filas de validación. El modelo no ha aprendido nada; la evaluación ha sido engañada.

## Una variable que contiene el objetivo

Una columna puede estar en la tabla y aun así no estar disponible cuando se hace la predicción: se calculó a partir del objetivo, o se registró después del suceso que se predice. Supón que los distritos de California vinieran con el impuesto sobre bienes inmuebles que pagan sus propietarios, que se calcula a partir del valor de la vivienda, alrededor del 1 %, con cierta variación. Un modelo entrenado con él:

In [ ]:
from sklearn.datasets import fetch_california_housing
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import train_test_split

housing = fetch_california_housing(as_frame=True).frame
X = housing.drop(columns="MedHouseVal")
y = 100_000 * housing["MedHouseVal"]
X_with_tax = X.assign(property_tax=0.01 * y * rng.normal(1, 0.05, len(y)))
X_train, X_test, y_train, y_test = train_test_split(X_with_tax, y, test_size=0.2, random_state=42)

with_tax = HistGradientBoostingRegressor(random_state=42).fit(X_train, y_train)
without_tax = HistGradientBoostingRegressor(random_state=42).fit(X_train.drop(columns="property_tax"), y_train)
{
    "with the tax": mean_absolute_error(y_test, with_tax.predict(X_test)),
    "without it": mean_absolute_error(y_test, without_tax.predict(X_test.drop(columns="property_tax"))),
}

{'with the tax': 7738.343639895265, 'without it': 31105.521289073844}

Con el impuesto, el modelo se equivoca en unos 7700 \$; sin él, en unos 31 100 \$. El impuesto es el 1 % del valor con un poco de variación, así que la columna es casi la respuesta: el modelo solo tiene que leerla. Pero una casa se valora antes de que se haya fijado un impuesto sobre ese valor: en producción la columna no existiría, y el modelo, que aprendió a depender de ella, no tendría nada a lo que recurrir. Lo mismo ocurre siempre que una columna se calcula a partir del objetivo (un precio por metro cuadrado, cuando lo que se predice es el precio) o se registra después del resultado (un tratamiento dado después de un diagnóstico, un reembolso emitido después de un fraude).

## El mismo ejemplo a ambos lados

Cuando la misma entidad aparece en varias filas, un paciente visitado dos veces, una casa vendida tres veces, un mensaje enviado muchas veces, una división aleatoria pone copias suyas a ambos lados, y la prueba mide en parte si el modelo la recuerda. Supón que cada uno de 3000 distritos aparece tres veces, con valores que difieren unos pocos puntos porcentuales, como harían anuncios repetidos del mismo lugar. Pliegues aleatorios frente a pliegues que mantienen juntas todas las copias de un distrito (`GroupKFold`):

In [ ]:
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import GroupKFold

districts = housing.sample(3000, random_state=0).reset_index(drop=True)
copies = pd.concat([districts.assign(district=districts.index)] * 3, ignore_index=True)
copies["MedHouseVal"] *= rng.normal(1, 0.03, len(copies))
X_copies = copies.drop(columns=["MedHouseVal", "district"])
y_copies = 100_000 * copies["MedHouseVal"]
forest = RandomForestRegressor(n_estimators=100, random_state=0, n_jobs=-1)
{
    "random folds": -cross_val_score(forest, X_copies, y_copies, cv=KFold(5, shuffle=True, random_state=0),
                                     scoring="neg_mean_absolute_error").mean(),
    "folds by district": -cross_val_score(forest, X_copies, y_copies, cv=GroupKFold(5), groups=copies["district"],
                                          scoring="neg_mean_absolute_error").mean(),
}

{'random folds': np.float64(11679.135183869794),
 'folds by district': np.float64(40264.298905102325)}

Con pliegues aleatorios el bosque parece equivocarse en unos 11 700 \$; con las copias de cada distrito juntas, en unos 40 300 \$, más de tres veces más. En los pliegues aleatorios, casi todos los distritos de validación tienen una copia en los pliegues de entrenamiento, y el bosque la encuentra y repite su valor. Un distrito nuevo, el tipo que traerá la producción, no tiene copia en ninguna parte. El error honesto también es mayor que el del bosque con los datos completos (unos 33 600 \$ en [Validación y selección de modelos](validation_and_model_selection.ipynb)), porque estos pliegues solo tienen 3000 distritos distintos de los que aprender.

## Información del futuro

Cuando un modelo vaya a predecir el futuro, el pasado es todo lo que tendrá. Una división aleatoria le da filas posteriores a aquellas con las que se le prueba. Cinco años de ventas simuladas de pisos, con precios que suben alrededor de un 1 % al mes: el mismo modelo, probado una vez con un 20 % aleatorio de las ventas, y otra vez con el último 20 % en el tiempo, tras entrenar con todo lo anterior:

In [ ]:
n_sales = 3000
month = np.sort(rng.uniform(0, 60, n_sales))
size = rng.uniform(50, 150, n_sales)
price = 2000 * size * (1 + 0.01 * month) * rng.normal(1, 0.08, n_sales)
sales = pd.DataFrame({"size_m2": size, "month": month, "price": price})

model = HistGradientBoostingRegressor(random_state=0)
random_train, random_test = train_test_split(sales, test_size=0.2, random_state=0)
cut = int(0.8 * n_sales)
past, future = sales.iloc[:cut], sales.iloc[cut:]
{
    "random 20%": mean_absolute_error(random_test["price"], model.fit(random_train[["size_m2", "month"]], random_train["price"])
                                      .predict(random_test[["size_m2", "month"]])),
    "last 20% in time": mean_absolute_error(future["price"], model.fit(past[["size_m2", "month"]], past["price"])
                                            .predict(future[["size_m2", "month"]])),
}

{'random 20%': 16899.887452767074, 'last 20% in time': 22304.700528280682}

Probado con un 20 % aleatorio de las ventas, el modelo se equivoca en unos 16 900 \$; probado con el último año, en unos 22 300 \$, un 30 % más. Con filas aleatorias, cada mes de prueba tiene ventas de entrenamiento justo antes y después, así que el modelo nunca tiene que mirar más allá de los precios que ha visto. En el último año tiene que predecir precios más altos que cualquiera de los que muestran sus filas de entrenamiento para un piso de ese tamaño, y los árboles no pueden extrapolar: predicen, como mucho, los precios de los últimos meses que vieron. Una división aleatoria de datos que se desplazan con el tiempo mide un modelo que ha visto el futuro; [King County: fuga temporal](https://github.com/avidaldo/king-county/blob/master/03-temporal_leakage.ipynb) muestra lo mismo con ventas de casas reales.

## Evitarla

| Tipo | Qué se filtra | Cómo se evita |
| --- | --- | --- |
| Aprender de las filas de validación o de prueba | sus estadísticos, a través de un escalador, un imputador, una selección | todo paso que aprende va dentro de un pipeline, ajustado solo con las filas de entrenamiento |
| Una variable que contiene el objetivo | la respuesta, a través de una columna calculada a partir de él o registrada después | pregúntate de cada columna: ¿existiría en el momento de la predicción? |
| El mismo ejemplo a ambos lados | la memoria de filas concretas | dividir por entidad (`GroupKFold`), tras eliminar los duplicados exactos |
| Información del futuro | filas posteriores | dividir por tiempo (`TimeSeriesSplit`, o entrenar con el pasado y probar con lo más reciente) |
| Elegir con el conjunto de prueba | la puntuación de prueba, a través de la decisión | un conjunto de validación o validación cruzada para cada decisión; el conjunto de prueba, una vez |

La sospecha es el mejor detector: una puntuación mucho mejor de lo esperado, o una sola variable mucho más importante que el resto, merece una comprobación antes que una celebración.